<h1 align = "center">
Rwanda Maize-Urea Trial
</h1>

<h2 align = "center">
Fertilizer Advisory Analysis
</h2> 

## Objective

We want to now turn the Rwanda maize trial results into a practical decision-support framework for Evergreen Institute. We want to identify which Urea application strategy offers the strongest yield evidence, whether the results differ meaningfully across growing environments, and what farmers would need to consider before adopting a recommendation.

Our central question is:

Which of the four Urea rate-and-timing strategies should Evergreen Institute consider recommending to farmers, given the observed maize yields and the statistical evidence?

#### Imports and Setup

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
from pathlib import Path

# Define path to folder
data_path = Path(r"../01_Data/processed")

def load_csv(filename):
    return pd.read_csv(data_path/filename)

In [3]:
plot_with_field_soil = load_csv("plot_with_field_soil.csv")
plot_with_field_soil.head()

,block_id,season,plot_id,treat_code,treat_details,urea_rate_timing,crop,prior_season_crop,planting_date,compost_tha,...,check_planting_date_before_harvest_date,days_silking_to_R3,calculated_urea_total,urea_total_difference,meg_env,_merge,pH,soil_match,has_soil_record,has_field_match
0,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-A,A,DAP 1Kg/Are + Urea 0.5Kg/Are at V6,Urea 0.5Kg/Are(V6),maize,maize,9/26/2020,8,...,False,20.0,50,0,Low-Mid-Dry,both,5.21,both,True,True
1,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-B,B,DAP 1Kg/Are + Urea 0.5Kg/Are at Planting and U...,Urea1kg/Are(Planting+V6),maize,maize,9/26/2020,8,...,False,20.0,100,0,Low-Mid-Dry,both,NaN,left_only,False,True
2,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-C,C,DAP 1Kg/Are + Urea 0.5Kg/Are at V6 + Urea 0.5K...,Urea 1kg/Are(V6 + V10),maize,maize,9/26/2020,8,...,False,20.0,100,0,Low-Mid-Dry,both,NaN,left_only,False,True
3,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-D,D,DAP 1Kg/Are at planting+ Urea 1Kg/Are at V6,Urea 1kg/Are(V6),maize,maize,9/26/2020,8,...,False,20.0,100,0,Low-Mid-Dry,both,NaN,left_only,False,True
4,d8807f25-e675-49ae-87a3-d228eca23c72,21A,d8807f25-e675-49ae-87a3-d228eca23c72-21A-A,A,DAP 1Kg/Are + Urea 0.5Kg/Are at V6,Urea 0.5Kg/Are(V6),maize,maize,9/26/2020,10,...,False,20.0,50,0,Low-Mid-Dry,both,5.75,both,True,True


In [5]:
import statsmodels.formula.api as smf

# Keep plots with the field context needed for this model.
# Five plots lack a matching mega-environment.
analysis_df = plot_with_field_soil.dropna(
    subset=["meg_env"]
).copy()

# Identify each field within its season.
analysis_df["field_season_id"] = (
    analysis_df["block_id"].astype(str)
    + "_"
    + analysis_df["season"].astype(str)
)

# Set the 50 kg/ha treatment as the reference category.
treatment_order = [
    "Urea 0.5Kg/Are(V6)",
    "Urea1kg/Are(Planting+V6)",
    "Urea 1kg/Are(V6)",
    "Urea 1kg/Are(V6 + V10)"
]

analysis_df["urea_rate_timing"] = pd.Categorical(
    analysis_df["urea_rate_timing"],
    categories=treatment_order
)

# Fit the baseline mixed-effects model.
model = smf.mixedlm(
    "yield_tha ~ C(urea_rate_timing) + C(season) + C(meg_env)",
    data=analysis_df,
    groups=analysis_df["field_season_id"]
)

model_result = model.fit(reml=True)

#### What we already know

The trial compares four strategies:

|Treatment	|Urea rate	|Application strategy|
|----|----|----|
|A	|50 kg/ha	|All at V6|
|B	|100 kg/ha	|Split between planting and V6|
|C	|100 kg/ha	|Split between V6 and V10|
|D	|100 kg/ha	|All at V6|


Our earlier analysis produced the following initial evidence:

- Treatment D (100 kg/ha, all at V6) had the highest overall observed mean yield, approximately 5.11 t/ha.

- Treatment C (100 kg/ha, split between V6 and V10) had an observed mean of approximately 5.06 t/ha, although its results were strongly affected by the 50.1 t/ha observation.

- Treatment B (100 kg/ha, split between planting and V6) had an observed mean of approximately 4.71 t/ha.

- Treatment A (50 kg/ha, all at V6) had an observed mean of approximately 4.57 t/ha.

But we know these are only descriptive averages, not yet sufficient grounds for recommending one treatment. 

- Differences in season and growing environment can influence yield, so we need to base our advice on the adjusted statistical results as well.

Our earlier mixed-effects model found that Treatments C and D yielded significantly more than Treatment A at the 5% significance level. 

- Treatment B's difference from Treatment A was not statistically significant in that initial model.

We also found no statistically significant evidence that treatment effects varied by mega-environment or season. This does not prove that the effects are identical everywhere; it means the available evidence was insufficient to establish those interactions at the 5% level.

Our first task is to produce a clear table of adjusted treatment yields and pairwise comparisons. This will give us a more defensible basis for the advisory analysis.

#### Adjusted treatment yields

- We'll use the original mixed-effects model, which adjusts for season and mega-environment and accounts for clustering by field-season.

- We want to calculate model-adjusted mean yields for each treatment, averaging over the observed season and mega-environment distribution. 

- We also want to calculate confidence intervals and pairwise treatment comparisons.

- We'd rather do this than compare raw means alone because we can now account for the season and mega-environment terms in the model.

In [6]:
# Calculate adjusted yields and confidence intervals
import numpy as np
import pandas as pd
from patsy import build_design_matrices
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

# Fixed-effect coefficients and their covariance matrix
fe_params = model_result.fe_params
fe_names = fe_params.index

cov_all = model_result.cov_params()
cov_fe = cov_all.loc[fe_names, fe_names]

# Retrieve the design information from the fitted model
design_info = model_result.model.data.design_info

# Calculate adjusted means for each treatment
adjusted_means = []

for treatment in treatment_order:

    new_data = analysis_df.copy()

    new_data["urea_rate_timing"] = pd.Categorical(
        [treatment] * len(new_data),
        categories=treatment_order
    )

    # Create the model design matrix for this treatment
    X = np.asarray(
        build_design_matrices(
            [design_info],
            new_data
        )[0]
    )

    # Average the design matrix across observed seasons and MEs
    X_mean = X.mean(axis=0)

    # Estimated adjusted mean yield
    estimate = X_mean @ fe_params.to_numpy()

    # Standard error, accounting for covariance among coefficients
    variance = X_mean @ cov_fe.to_numpy() @ X_mean.T
    se = np.sqrt(max(variance, 0))

    adjusted_means.append({
        "treatment": treatment,
        "adjusted_mean_yield": estimate,
        "std_error": se,
        "ci_lower": estimate - 1.96 * se,
        "ci_upper": estimate + 1.96 * se
    })

adjusted_means_df = pd.DataFrame(adjusted_means)

adjusted_means_df.round(3)

,treatment,adjusted_mean_yield,std_error,ci_lower,ci_upper
0,Urea 0.5Kg/Are(V6),4.571,0.102,4.372,4.770
1,Urea1kg/Are(Planting+V6),4.715,0.102,4.516,4.915
2,Urea 1kg/Are(V6),5.117,0.102,4.918,5.316
3,Urea 1kg/Are(V6 + V10),5.071,0.102,4.872,5.270


**We want to now Compare every pair of treatments**

We know that a treatment can have a higher adjusted mean than another without the difference being statistically significant. Let's calculate the pairwise differences and adjust for multiple comparisons.

In [7]:
pairwise_results = []

# Recreate the average design matrix for each treatment
mean_design_matrices = {}

for treatment in treatment_order:

    new_data = analysis_df.copy()

    new_data["urea_rate_timing"] = pd.Categorical(
        [treatment] * len(new_data),
        categories=treatment_order
    )

    X = np.asarray(
        build_design_matrices(
            [design_info],
            new_data
        )[0]
    )

    mean_design_matrices[treatment] = X.mean(axis=0)

# Compare each unique pair
for i, treatment_a in enumerate(treatment_order):

    for treatment_b in treatment_order[i + 1:]:

        contrast = (
            mean_design_matrices[treatment_a]
            - mean_design_matrices[treatment_b]
        )

        difference = contrast @ fe_params.to_numpy()

        variance = (
            contrast
            @ cov_fe.to_numpy()
            @ contrast.T
        )

        se = np.sqrt(max(variance, 0))

        z_stat = difference / se if se > 0 else np.nan

        p_value = (
            2 * norm.sf(abs(z_stat))
            if np.isfinite(z_stat)
            else np.nan
        )

        pairwise_results.append({
            "treatment_a": treatment_a,
            "treatment_b": treatment_b,
            "difference_t_ha": difference,
            "std_error": se,
            "ci_lower": difference - 1.96 * se,
            "ci_upper": difference + 1.96 * se,
            "p_value": p_value
        })

pairwise_df = pd.DataFrame(pairwise_results)

# Holm correction controls the family-wise error rate
valid = pairwise_df["p_value"].notna()

pairwise_df["p_value_holm"] = np.nan
pairwise_df["significant_5pct"] = False

if valid.any():

    adjusted_p = multipletests(
        pairwise_df.loc[valid, "p_value"],
        method="holm"
    )[1]

    pairwise_df.loc[valid, "p_value_holm"] = adjusted_p

    pairwise_df.loc[valid, "significant_5pct"] = (
        adjusted_p < 0.05
    )

pairwise_df.round(4)

,treatment_a,treatment_b,difference_t_ha,std_error,ci_lower,ci_upper,p_value,p_value_holm,significant_5pct
0,Urea 0.5Kg/Are(V6),Urea1kg/Are(Planting+V6),-0.1443,0.1021,-0.3444,0.0558,0.1576,0.3152,False
1,Urea 0.5Kg/Are(V6),Urea 1kg/Are(V6),-0.5458,0.1019,-0.7454,-0.3462,0.0000,0.0000,True
2,Urea 0.5Kg/Are(V6),Urea 1kg/Are(V6 + V10),-0.5000,0.1019,-0.6997,-0.3004,0.0000,0.0000,True
3,Urea1kg/Are(Planting+V6),Urea 1kg/Are(V6),-0.4015,0.1021,-0.6016,-0.2014,0.0001,0.0003,True
4,Urea1kg/Are(Planting+V6),Urea 1kg/Are(V6 + V10),-0.3558,0.1021,-0.5559,-0.1557,0.0005,0.0015,True
5,Urea 1kg/Are(V6),Urea 1kg/Are(V6 + V10),0.0458,0.1019,-0.1539,0.2454,0.6532,0.6532,False


## Results Interpretation

These results give us a much clearer basis for the advisory analysis. The key finding is that applying 100 kg/ha of Urea entirely at V6 produced the highest adjusted mean yield, but splitting the same rate between V6 and V10 produced a statistically comparable yield.

**So what does this mean for farmers?**

****A. Adjusted yield performance**

Our model-adjusted means are:

|Treatment	|Adjusted yield (t/ha)	|95% confidence interval|
|----|---:|---:|
|50 kg/ha, all at V6	|4.571	|4.372–4.770|
|100 kg/ha, split between planting and V6	|4.715	|4.516–4.915|
|100 kg/ha, all at V6	|5.117	|4.918–5.316|
|100 kg/ha, split between V6 and V10	|5.071	|4.872–5.270|


The estimated difference between the highest- and lowest-yielding treatments is approximately 0.546 tonnes of maize per hectare.

That is a potentially useful yield difference. However, we must distinguish the size of the estimated difference from whether it is statistically supported and economically worthwhile.

##### **Which treatment differences are statistically supported?**

Our pairwise comparisons use the Holm correction to account for the six comparisons among the four treatments.

|Comparison	|Estimated yield difference	|Holm-adjusted p-value	|Statistically significance at 5%|
|----|---:|---:|----|
|100 kg/ha, all at V6 vs 50 kg/ha, all at V6	|+0.546 t/ha	|<0.001	|Statistically significant|
|100 kg/ha, split V6–V10 vs 50 kg/ha, all at V6	|+0.500 t/ha	|<0.001	|Statistically significant|
|100 kg/ha, split planting–V6 vs 50 kg/ha, all at V6	|+0.144 t/ha	|0.3152	|Not statistically significant|
|100 kg/ha, all at V6 vs split planting–V6	|+0.402 t/ha	|0.0003	|Statistically significant|
|100 kg/ha, split V6–V10 vs split planting–V6	|+0.356 t/ha	|0.0015	|Statistically significant|
|100 kg/ha, all at V6 vs split V6–V10	|+0.046 t/ha	|0.6532	|Not statistically significant|


**Three important conclusions**

1. The two strongest yield-performing treatments outperform the 50 kg/ha treatment.

    - Both applying 100 kg/ha entirely at V6 and splitting 100 kg/ha between V6 and V10 produce significantly higher adjusted yields than applying 50 kg/ha entirely at V6.

2. Splitting the 100 kg/ha rate between planting and V6 performs less well in this trial.

    - Its adjusted yield is significantly lower than either of the other 100 kg/ha strategies. Its difference from the 50 kg/ha treatment is not statistically significant.

    - This does not establish that applying Urea at planting is inherently harmful. The result applies to these treatment combinations under the conditions represented in this trial.

3. We cannot establish that applying all the Urea at V6 is better than splitting it between V6 and V10.

    - The estimated advantage is only 0.046 t/ha, and the adjusted p-value is 0.6532. The data do not provide sufficient evidence of a difference between these two strategies.

This is an important advisory distinction: the treatment with the highest estimated yield is not necessarily statistically superior to the second-highest treatment.

#### **Does the recommendation change across mega-environments or seasons?**

We previously tested whether the effect of treatment varied across mega-environments and seasons.

|Interaction tested	|Likelihood-ratio statistic	|Degrees of freedom	|p-value	|Conclusion|
|----|----|----|----|----|
|Treatment × mega-environment	|9.142	|9	|0.424	|No statistically significant interaction detected|
|Treatment × season	|10.564	|6	|0.103	|No statistically significant interaction detected|


At the 5% significance level, neither interaction is statistically significant.

This means we do not currently have sufficient evidence to recommend different Urea treatments for different mega-environments or seasons based on differences in treatment response.

It does not, however, prove that every environment responds identically. 

- Environmental differences may still affect overall yield, and 

- The study may have limited power to detect treatment interactions.

**What this means for the advisory tool**

For now, we just want to develop a general, trial-based recommendation rather than separate treatment rules for Highlands, Low-Mid-Dry, Low-Mid-Wet, and Transitional environments.

The tool can still display the expected yields by mega-environment to help users understand local context. However, it should not imply that a treatment is statistically superior in a particular environment when the interaction analysis does not support that conclusion.

#### **Yield Results Translation into Practical Advice**

We can now formulate a provisional recommendation based on yield performance.

|Advisory category	|Treatment	|Evidence-based interpretation|
|----|----|----|
|Primary yield-oriented candidate	|100 kg/ha, all at V6	|Highest adjusted yield; significantly outperforms the 50 kg/ha treatment and the planting–V6 split.|
|Alternative yield-oriented candidate	|100 kg/ha, split between V6 and V10	|Yield is statistically comparable to applying all 100 kg/ha at V6; significantly outperforms the 50 kg/ha treatment and the planting–V6 split.|
|Lower-input option	|50 kg/ha, all at V6	|Lower adjusted yield, but uses 50 kg/ha less Urea than the full-rate treatments. Its economic attractiveness remains to be assessed.|
|Less promising yield option in this trial	|100 kg/ha, split between planting and V6	|Lower adjusted yield than the other 100 kg/ha strategies; its advantage over the 50 kg/ha treatment was not statistically significant.|


These are provisional options, not yet final recommendations for farmers.

There are practical considerations to address. For example, splitting the dose between V6 and V10 requires a second application later in crop development. The timing, availability of labor, and feasibility of applying fertilizer at V10 could influence whether that strategy is suitable for a particular farmer.

We should also remember the missing V10 application dates in part of the dataset. That issue does not invalidate the yield comparison automatically, but it limits what we can infer about the timing and implementation of those applications.

### **Final advisory conclusions**

For Evergreen Institute, our findings can be summarized follows:

|Finding	|Evidence	|Implication for advisory decisions|
|----|----|----|
|Applying 100 kg/ha entirely at V6 produced the highest adjusted mean yield.	|5.117 t/ha	|Leading candidate when the objective is to maximize yield.|
|Splitting 100 kg/ha between V6 and V10 produced a comparable yield.	|5.071 t/ha; adjusted pairwise p = 0.6532	|A reasonable alternative for further consideration.|
|Both treatments outperformed the 50 kg/ha treatment in adjusted yield.	|Yield advantages of 0.546 and 0.500 t/ha, respectively; both statistically significant after Holm correction	|The trial supports a yield advantage from these full-rate strategies under the conditions studied.|
|Splitting 100 kg/ha between planting and V6 performed less well than the other full-rate strategies.	|Adjusted mean of 4.715 t/ha	|Less promising when the sole objective is yield.|
|No statistically significant treatment interactions with mega-environment or season were detected.	|Interaction p-values of 0.424 and 0.103	|The available evidence does not justify separate treatment recommendations by mega-environment or season.|
